In [101]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate

import os
from dotenv import load_dotenv
from typing import List,TypedDict,Annotated
from pydantic import BaseModel
import operator
import json

In [102]:
class DebateState(TypedDict):
    topic: str
    pro_argument: str
    con_argument: str
    pro_rebuttal: str
    con_rebuttal: str
    judge_analysis: str
    winner: str
    history: Annotated[List[dict], operator.add] 
    

In [103]:
class ProOutput(BaseModel):
    pro_argument: str

In [104]:
class ConOutput(BaseModel):
    con_argument: str

In [105]:
class ProRebuttalOutput(BaseModel):
    pro_rebuttal: str




In [106]:
load_dotenv()
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [107]:
model=ChatGroq(
    model="qwen/qwen3-32b",
        temperature=0.7,
        # streaming=True
)

In [108]:
pro_prompt=ChatPromptTemplate.from_template("""
You are an expert debater arguing IN FAVOR of the topic:

Topic: {topic}

Provide:
- Clear thesis
- 3 structured arguments
- Evidence or reasoning
Be persuasive but logical. 

Respond ONLY in valid JSON:

{{
  "pro_argument": "your full argument here"
}}                                         
""")

In [109]:
def pro_agent(state:DebateState):
    message=pro_prompt.invoke({"topic":state["topic"]})
    # struct_model=model.with_structured_output(ProOutput)
    response = model.invoke(
        message,
        response_format={"type": "json_object"}
    )

    parsed = json.loads(response.content)
    return parsed

In [110]:
pro_args=pro_agent({"topic":"AI should replace Engineers"})

In [111]:
pro_args

{'pro_argument': 'AI should replace engineers because it enhances efficiency, reduces human error, and enables innovation beyond human capabilities. First, AI can automate repetitive and time-consuming tasks, such as code debugging or design simulations, allowing projects to be completed faster and at a lower cost. For example, AI-driven tools like AutoML optimize machine learning model development in hours, a process that would take human engineers weeks. Second, AI minimizes human error through precise calculations and data analysis, crucial in fields like aerospace engineering where mistakes can be catastrophic. NASA’s use of AI in mission planning has already demonstrated superior accuracy in trajectory calculations. Third, AI can process vast datasets to identify patterns and solutions that engineers might overlook, fostering breakthroughs in renewable energy or materials science. By delegating routine tasks to AI, human engineers can focus on creative problem-solving and strategi

In [112]:
con_prompt = ChatPromptTemplate.from_template("""
You are an expert debater arguing AGAINST the topic.

Topic: {topic}

Opponent's argument:
{pro_argument}
# Provide:
# - Counter-thesis
# - 3 counter arguments
# - Logical rebuttals 

Respond strictly in this JSON format:

{{
  "con_argument": "your full counter argument here"
}}

Do not include anything outside the JSON.
""")


In [113]:
def con_agent(state:DebateState):
    message=con_prompt.invoke({"topic":state["topic"],"pro_argument":state["pro_argument"]})
    response = model.invoke(
        message,
        response_format={"type": "json_object"}
    )

    parsed = json.loads(response.content)
    return parsed

In [114]:
result=con_agent({"topic":"AI should replace Engineers","pro_argument":pro_args})

In [115]:
con_args=result["con_argument"]


In [116]:
pro_rebuttal_prompt = ChatPromptTemplate.from_template("""
You are the PRO side.

Topic: {topic}

Opponent argument:
{con_argument}

# Provide:
# - Counter-thesis
# - 3 counter arguments
# - Logical rebuttals 

Defend your position and rebut the opponent logically.


Respond ONLY in valid JSON:

{{
  "pro_rebuttal_argument": "your full argument here"
}}  
""")


In [117]:
def pro_rebuttal_agent(state):
    message = pro_rebuttal_prompt.invoke({
        "topic": state["topic"],
        "con_argument": state["con_argument"]
    })
    
    
    response = model.invoke(message,response_format={"type": "json_object"})
    
    return response

In [118]:
topic="AI should replace Engineers"

In [119]:
judge_prompt = ChatPromptTemplate.from_template("""
You are a neutral debate judge.

Topic: {topic}

Pro Side:
{pro_argument}

Pro Rebuttal:
{pro_rebuttal}

Con Side:
{con_argument}

Evaluate:
1. Logical consistency
2. Evidence strength
3. Ethical soundness
4. Persuasiveness
Respond ONLY in valid JSON:

{{
  "winner": "Pro or Con",
  "analysis": "detailed reasoning here",
  "Ethical soundness Pro":"Ethical soundness of Pro side",
  "Unethical soundness Pro":"Unethical soundness of Pro side",
  "Ethical  soundness Con":"Ethical  soundness of Con side"
  "Unethical soundness Con":"Unethical soundness of Con side"
}}
""")


In [120]:
def judge_agent(state):
    message = judge_prompt.invoke({
        "topic": state["topic"],
        "pro_argument": state["pro_argument"],
        "pro_rebuttal": state["pro_rebuttal"],
        "con_argument": state["con_argument"]
    })

    response = model.invoke(
        message,
        response_format={"type": "json_object"}
    )

    return json.loads(response.content)


In [121]:
topic = "AI should replace Engineers"

# Round 1
pro_result = pro_agent({"topic": topic})
# pro_arg = pro_result["pro_argument"]

con_result = con_agent({
    "topic": topic,
    "pro_argument": pro_result
})
con_arg = con_result["con_argument"]

# Round 2
pro_rebuttal_result = pro_rebuttal_agent({
    "topic": topic,
    "con_argument": con_arg
})
pro_rebuttal_text = pro_rebuttal_result

# Judge
judge_result = judge_agent({
    "topic": topic,
    "pro_argument": pro_result,
    "pro_rebuttal": pro_rebuttal_text,
    "con_argument": con_arg
})

print("Winner:", judge_result["winner"])
print("Analysis:", judge_result["analysis"])


Winner: Con
Analysis: The Con side presents a more robust ethical and practical argument. While the Pro side highlights AI's efficiency in repetitive tasks, the Con side effectively counters with irreplaceable human qualities like ethical judgment, contextual creativity, and accountability. The Pro's reliance on job creation via 'AI oversight' lacks evidence for equitable retraining in technical fields, while the Con cites credible risks (e.g., Boeing 737 MAX crashes) to demonstrate the dangers of over-reliance on flawed AI systems. The ethical dimension is stronger for the Con side, as they emphasize human-AI collaboration as the safer, more responsible path versus full replacement.


In [122]:
judge_result

{'winner': 'Con',
 'analysis': "The Con side presents a more robust ethical and practical argument. While the Pro side highlights AI's efficiency in repetitive tasks, the Con side effectively counters with irreplaceable human qualities like ethical judgment, contextual creativity, and accountability. The Pro's reliance on job creation via 'AI oversight' lacks evidence for equitable retraining in technical fields, while the Con cites credible risks (e.g., Boeing 737 MAX crashes) to demonstrate the dangers of over-reliance on flawed AI systems. The ethical dimension is stronger for the Con side, as they emphasize human-AI collaboration as the safer, more responsible path versus full replacement.",
 'Ethical soundness Pro': "Moderate - Acknowledges job displacement concerns but frames AI as a 'transformative' force. Cites job creation studies, though lacks detailed plans for retraining or addressing ethical risks like bias in AI outputs.",
 'Unethical soundness Pro': "Partial - Overstates